In [1]:
import sys
import pandas as pd
from pathlib import Path

# Add src/ to path so notebook can import pipeline modules
sys.path.insert(0, str(Path("../src")))

In [10]:
import data_loading
import config
import classifier_data
from sklearn.model_selection import StratifiedShuffleSplit


# fix these imports lol
# import df replicates original
# import heatmap_matrix (probably best to just save it at end of run_pipeline.py to be able to import it).
# see claude here.
# Mask df replicates with this. -> shape (1000 probes, 268 replicates)
# Do we need to filter/merge this accordingly to df_merged? probably. Remove any tissues not in heatmap_matrix. What to do with the merged tissues??

# Use stratified k-fold CV instead of splitting into train/val/test
# train logistic model with this, should be quick.

In [3]:
#Note that heatmap_matrix has tissues as the index (row labels).
#When you load it back in the notebook you'll need pd.read_csv(path, index_col=0) to restore them correctly.

heatmap_matrix_path = config.DATA_DIR / "processed" / "top_50_regions_df.csv"

heatmap_matrix = pd.read_csv(heatmap_matrix_path, index_col=0)

heatmap_matrix

,cg46037634_TC11,cg45190349_TC11,cg30114922_TC11,cg45888439_TC11,cg43303701_TC11,cg31153580_TC11,cg38472839_TC21,cg43753330_TC11,cg44021726_TC21,cg38576506_BC21,...,cg43562992_TC11,cg31908281_TC11,cg43913360_BC11,cg36976410_TC11,cg42519592_BC21,cg34885304_TC11,cg43673861_BC11,cg30039856_BC11,cg33267424_BC11,cg38298887_BC11
Eye_Retina,0.167157,0.246529,0.299543,0.301793,0.246543,0.245629,0.245043,0.351200,0.262864,0.259329,...,0.912879,0.873936,0.823764,0.836157,0.814900,0.913950,0.759143,0.883936,0.857164,0.769793
Brain_Cortex_Subcortical,0.866278,0.917806,0.934549,0.741139,0.792146,0.851073,0.814618,0.870889,0.867229,0.863000,...,0.905944,0.794344,0.870486,0.902062,0.845632,0.712795,0.924420,0.901722,0.881580,0.796892
Blood_Spleen_Thymus,0.924637,0.941307,0.939556,0.947943,0.907483,0.884761,0.866607,0.957869,0.862783,0.845216,...,0.923301,0.885943,0.901504,0.916883,0.849569,0.958332,0.932602,0.889952,0.895755,0.914415
Skin_Ears_Tail,0.774226,0.938631,0.919644,0.820967,0.790429,0.850568,0.818764,0.953016,0.741150,0.803225,...,0.707628,0.814682,0.853258,0.624458,0.798823,0.767057,0.900348,0.852117,0.783076,0.734602
Adrenal,0.891000,0.909500,0.888333,0.920333,0.793833,0.835167,0.793500,0.939000,0.841833,0.773333,...,0.873000,0.806667,0.806667,0.865833,0.805000,0.921333,0.854000,0.878333,0.828667,0.863333
Bile_Duct,0.907000,0.936333,0.908500,0.930833,0.894667,0.852667,0.816500,0.953500,0.825000,0.806000,...,0.903667,0.868833,0.736333,0.799333,0.812000,0.835333,0.902500,0.847500,0.856500,0.907500
Cerebellum,0.518333,0.957444,0.921000,0.897556,0.485444,0.880333,0.848667,0.881556,0.878222,0.874778,...,0.920111,0.821111,0.925444,0.904333,0.870333,0.819222,0.789667,0.915667,0.544444,0.847444
Colon,0.916455,0.934000,0.915364,0.930000,0.867636,0.811727,0.799273,0.926636,0.801091,0.781273,...,0.681545,0.854818,0.781727,0.871182,0.728364,0.856636,0.935091,0.822455,0.781818,0.610273
Diaphragm,0.856000,0.902500,0.911500,0.900333,0.846833,0.820833,0.765167,0.936833,0.813500,0.797833,...,0.862000,0.832167,0.807167,0.777500,0.797833,0.774833,0.873833,0.814667,0.805667,0.873667
Femur,0.868100,0.919900,0.869400,0.880600,0.867700,0.805300,0.808300,0.853200,0.845700,0.826400,...,0.811900,0.820300,0.800300,0.765100,0.830700,0.918900,0.869600,0.750100,0.770200,0.848900


In [4]:
# load raw replicate dataframe
df_raw, _, _, _ = data_loading.load_all()

In [ ]:
df_raw

df_raw_labels = df_raw.index.to_list()
#df_raw_labels.value_counts().sort_index()


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [11]:
# get sample columns and extract tissue labels (same logic as classifier_data.py)
sample_cols = df_raw.columns[df_raw.columns != "probe_ID"].tolist()
original_labels = pd.Series(
    [col.split(".")[0] for col in sample_cols],
    index=sample_cols,
)

# exclude dropped tissues before splitting
original_labels = original_labels[~original_labels.isin(data_loading.EXCLUDED_TISSUES)]
sample_cols_clean = original_labels.index.tolist()

# pick exactly 1 sample per class for the test set
test_cols = (
    original_labels
    .groupby(original_labels)
    .apply(lambda grp: grp.sample(n=1, random_state=42))
    .index.get_level_values(1)  # get the sample column names
    .tolist()
)

train_cols = [c for c in sample_cols_clean if c not in test_cols]

# sanity checks
print(f"Total samples:  {len(sample_cols_clean)}")
print(f"Test samples:   {len(test_cols)}")   # should be 20 (1 per tissue)
print(f"Train samples:  {len(train_cols)}")

# check 1 per class in test
test_labels = original_labels[test_cols]
print("\nTest set class counts (should all be 1):")
print(test_labels.value_counts().sort_index())

# check minimum class size in train (drives fold count)
train_labels = original_labels[train_cols]
print("\nTrain set class counts:")
print(train_labels.value_counts().sort_index())
print(f"\nMin class size in train: {train_labels.value_counts().min()}")

Total samples:  247
Test samples:   26
Train samples:  221

Test set class counts (should all be 1):
Adrenal              1
Bile_Duct            1
Blood                1
Brain_Cortex         1
Cerebellum           1
Colon                1
Diaphragm            1
Ears                 1
Eye                  1
Femur                1
Heart                1
Kidney               1
Liver                1
Lung                 1
Pancreas             1
Retina               1
Skin                 1
Spinal_Cord          1
Spleen               1
Stomach              1
Subcortical_Brain    1
Tail                 1
Testis               1
Thymus               1
Urinary_Bladder      1
Uterus               1
Name: count, dtype: int64

Train set class counts:
Adrenal               5
Bile_Duct             5
Blood                11
Brain_Cortex         15
Cerebellum            8
Colon                10
Diaphragm             5
Ears                  7
Eye                   9
Femur                 9
Heart     

In [6]:
X, y, le, merged_labels = classifier_data.load_classifier_data()

heatmap_matrix shape (tissues x probes): (20, 1000)
df_raw shape (probes x samples): (288655, 268)

Unique original tissue labels: ['Adrenal', 'Bile_Duct', 'Blood', 'Brain_Cortex', 'Cerebellum', 'Colon', 'Diaphragm', 'Ears', 'Eye', 'Femur', 'Heart', 'Kidney', 'Liver', 'Lung', 'Mammary_Glands', 'Optic_Nerve', 'Pancreas', 'Retina', 'Sciatic_Nerve', 'Skin', 'Spinal_Cord', 'Spleen', 'Stomach', 'Subcortical_Brain', 'Tail', 'Testis', 'Thymus', 'Urinary_Bladder', 'Uterus']
Excluded 20 samples from: {'Optic_Nerve', 'Mammary_Glands', 'Sciatic_Nerve'}
Remaining samples: 247

Sample counts per merged tissue class:
merged_tissue
Adrenal                      6
Bile_Duct                    6
Blood_Spleen_Thymus         30
Brain_Cortex_Subcortical    25
Cerebellum                   9
Colon                       11
Diaphragm                    6
Eye_Retina                  17
Femur                       10
Heart                       10
Kidney                      12
Liver                       26
Lun

In [7]:
with pd.option_context("display.max_columns", None,
                       "display.width", None):
        display(X.head(40))

probe_ID         cg46037634_TC11  cg45190349_TC11  cg30114922_TC11  \
Adrenal                    0.901            0.931            0.925   
Adrenal.1                  0.903            0.920            0.885   
Adrenal.2                  0.880            0.878            0.865   
Adrenal.3                  0.870            0.866            0.860   
Adrenal.4                  0.909            0.924            0.890   
Adrenal.5                  0.883            0.938            0.905   
Bile_Duct                  0.900            0.935            0.909   
Bile_Duct.1                0.911            0.923            0.888   
Bile_Duct.2                0.919            0.929            0.900   
Bile_Duct.3                0.903            0.946            0.923   
Bile_Duct.4                0.915            0.947            0.917   
Bile_Duct.5                0.894            0.938            0.914   
Blood                      0.917            0.925            0.920   
Blood.1                    0.930            0.916            0.935   
Blood.2                    0.926            0.932            0.964   
Blood.3                    0.924            0.934            0.933   
Blood.4                    0.919            0.947            0.950   
Blood.5                    0.938            0.979            0.945   
Blood.6                    0.939            0.934            0.937   
Blood.7                    0.939            0.940            0.935   
Blood.8                    0.943            0.945            0.960   
Blood.9                    0.937            0.944            0.963   
Blood.10                   0.928            0.957            0.949   
Blood.11                   0.923            0.951            0.950   
Brain_Cortex               0.848            0.859            0.924   
Brain_Cortex.1             0.892            0.887            0.960   
Brain_Cortex.2             0.855            0.900            0.936   
Brain_Cortex.3             0.850            0.917            0.961   
Brain_Cortex.4             0.885            0.936            0.932   
Brain_Cortex.5             0.850            0.921            0.929   
Brain_Cortex.6             0.831            0.923            0.916   
Brain_Cortex.7             0.880            0.866            0.916   
Brain_Cortex.8             0.864            0.903            0.935   
Brain_Cortex.9             0.873            0.908            0.953   
Brain_Cortex.10            0.867            0.937            0.953   
Brain_Cortex.11            0.901            0.885            0.938   
Brain_Cortex.12            0.880            0.931            0.971   
Brain_Cortex.13            0.906            0.919            0.941   
Brain_Cortex.14            0.887            0.864            0.942   
Brain_Cortex.15            0.867            0.852            0.947   

probe_ID         cg45888439_TC11  cg43303701_TC11  cg31153580_TC11  \
Adrenal                    0.941            0.832            0.881   
Adrenal.1                  0.935            0.831            0.861   
Adrenal.2                  0.918            0.757            0.812   
Adrenal.3                  0.898            0.770            0.753   
Adrenal.4                  0.918            0.792            0.863   
Adrenal.5                  0.912            0.781            0.841   
Bile_Duct                  0.940            0.906            0.891   
Bile_Duct.1                0.944            0.918            0.846   
Bile_Duct.2                0.938            0.876            0.828   
Bile_Duct.3                0.925            0.898            0.864   
Bile_Duct.4                0.923            0.907            0.847   
Bile_Duct.5                0.915            0.863            0.840   
Blood                      0.951            0.900            0.866   
Blood.1                    0.949            0.875            0.842   
Blood.2                    0.945            0.886            0.867   
Blood.3  

In [8]:
merged_labels.head()

Adrenal      Adrenal
Adrenal.1    Adrenal
Adrenal.2    Adrenal
Adrenal.3    Adrenal
Adrenal.4    Adrenal
Name: merged_tissue, dtype: object

In [9]:
le

LabelEncoder()